# Raw FlexDC Inference + FlexDC Validation Dashboard v2

This notebook is focused on the current **FlexDC raw** model trained on `newqos_plus_w2dense_v1`.

It runs the existing prediction/optimization/e2e scripts, but uses the new v2 optimizer/e2e wrapper so you can optionally restrict P/R to a local box around the W2 feasible band. It also includes two human-readable summary cells:

1. a concise start-vs-selected table with P/R/weights and predicted vs actual raw quantities;
2. a constants playground that recomputes the paper-form objective from raw simulator outputs for any beta/rho/psi/mu/gamma/delta you want to try.

In [ ]:
%pip install -q wandb pandas numpy scipy scikit-learn tqdm matplotlib tabulate openpyxl

## 1. Paths

Run this after cloning/pulling both repos in Colab. Edit `WORKSPACE_ROOT`, `COMDER_ROOT`, or `FLEXDC_ROOT` only if your folder names differ.

In [ ]:
from pathlib import Path
import os

# Colab default used in our notebooks.
WORKSPACE_ROOT = Path("/content/workspace")
COMDER_ROOT = WORKSPACE_ROOT / "comder-main"
AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
FLEXDC_ROOT = WORKSPACE_ROOT / "flexdc-sim"

DATA_DIR = AM_FLEXDC_ROOT / "data"
MODELS_DIR = AM_FLEXDC_ROOT / "models"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "raw_inference_v2_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_DIR:", TRAIN_DIR)
print("FLEXDC_ROOT:", FLEXDC_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

## 2. W&B login

Use online mode when you want trajectories/results logged. Use `disabled` for quick tests.

In [ ]:
USE_WANDB = True
WANDB_MODE = "online"   # "online", "offline", or "disabled"
WANDB_PROJECT = "flexdc-unified-inference"
WANDB_ENTITY = "amenon06-boston-university"

if USE_WANDB and WANDB_MODE != "disabled":
    import os, getpass, wandb
    os.environ.pop("WANDB_BASE_URL", None)
    os.environ["WANDB_MODE"] = WANDB_MODE
    if WANDB_MODE == "online":
        ok = False
        try:
            from google.colab import userdata
            key_from_secret = userdata.get("WANDB_API_KEY")
        except Exception:
            key_from_secret = None
        if key_from_secret:
            os.environ["WANDB_API_KEY"] = key_from_secret
            ok = wandb.login(key=key_from_secret, relogin=True, verify=True)
        else:
            try:
                ok = wandb.login(relogin=True, verify=True)
            except Exception:
                api_key = getpass.getpass("Paste W&B API key: ")
                os.environ["WANDB_API_KEY"] = api_key
                ok = wandb.login(key=api_key, relogin=True, verify=True)
        if not ok:
            raise RuntimeError("W&B login failed. Set WANDB_MODE='disabled' or provide a valid key.")
        print("W&B login verified.")
    else:
        print("W&B offline mode.")
else:
    print("W&B disabled.")

## 3. Model + dataset controls

This notebook defaults to the latest raw FlexDC model and augmented dataset.

Raw target order is:

`[flexdc_M_RSR, raw_Ctrack_Epsilon_90th, raw_qos_probability_mean]`

The optimizer objective is therefore a **raw heuristic score**, not the exact paper objective. You can change the weights below to emphasize tracking or QoS.

In [ ]:
TARGET_FAMILY = "flexdc"
TARGET_MODE = "raw"
RAW_QOS_AGGREGATION = "mean"
USE_NORM_COST = "auto"
USE_NORM_PR = "true"
DEVICE = "cuda"  # "cuda", "cpu", or "auto"

DATASET_TAG = "newqos_plus_w2dense_v1"
PILOT_DIR = DATA_DIR / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective"
RESULTS_CSV = PILOT_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = PILOT_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv"

MODEL_FILE_CANDIDATES = [
    MODELS_DIR / "flexdc_raw" / "am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt",
    MODELS_DIR / "flexdc_raw" / "am_flexdc_raw_newqos_plus_w2dense_v1_state_dict.pt",
    MODELS_DIR / "am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt",
]
MODEL_FILE = next((p for p in MODEL_FILE_CANDIDATES if p.exists()), MODEL_FILE_CANDIDATES[0])

for p in [RESULTS_CSV, DIAGNOSTICS_CSV, MODEL_FILE]:
    print(p, "exists=", p.exists())

if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"Missing RESULTS_CSV: {RESULTS_CSV}")
if not DIAGNOSTICS_CSV.exists():
    raise FileNotFoundError(f"Missing DIAGNOSTICS_CSV: {DIAGNOSTICS_CSV}")
if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Missing model checkpoint. Expected one of: {MODEL_FILE_CANDIDATES}")

## 4. Copy/check v2 inference scripts

Place these two files in `am_flexdc/train` before running this cell:

- `am_unified_optimize_one_v2.py`
- `am_unified_end_to_end_eval_raw_v2.py`

They add explicit local P/R bounds and extra raw/QoS diagnostics while keeping the original model logic.

In [ ]:
required = [
    TRAIN_DIR / "data_center_model.py",
    TRAIN_DIR / "am_unified_training_utilities.py",
    TRAIN_DIR / "am_unified_predict_one.py",
    TRAIN_DIR / "am_unified_optimize_one_v2.py",
    TRAIN_DIR / "am_unified_end_to_end_eval_raw_v2.py",
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:
" + "
".join(str(p) for p in missing))

%cd {TRAIN_DIR}
!python -m py_compile am_unified_predict_one.py am_unified_optimize_one_v2.py am_unified_end_to_end_eval_raw_v2.py

## 5. Choose a validation case

Default is W2-LU around the Fatih-confirmed feasible region. Switch `CASE = "w2_hu"` if needed.

In [ ]:
CASE = "w2_lu"   # "w2_lu" or "w2_hu"

case_configs = {
    "w2_lu": {
        "workload_config": FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini",
        "experiment_config": FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini",
        "server_count": 1000,
        "utilization": 0.60,
        "start_pbar": 0.472128,
        "start_r": 0.102206,
        "start_weights": "0.2580196175290311,0.2508946902088517,0.25269483018214395,0.2383908620799734",
        "pbar_min": 0.464,
        "pbar_max": 0.480,
        "r_min": 0.075,
        "r_max": 0.140,
    },
    "w2_hu": {
        "workload_config": FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5555.ini",
        "experiment_config": FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini",
        "server_count": 1000,
        "utilization": 0.80,
        "start_pbar": 0.576789,
        "start_r": 0.138748,
        "start_weights": "0.25881071977840475,0.2545059365273737,0.24568691042720164,0.24099643326701997",
        "pbar_min": 0.570,
        "pbar_max": 0.593,
        "r_min": 0.110,
        "r_max": 0.180,
    },
}

cfg = case_configs[CASE]
WORKLOAD_CONFIG = cfg["workload_config"]
EXPERIMENT_CONFIG = cfg["experiment_config"]
SERVER_COUNT = cfg["server_count"]
UTILIZATION = cfg["utilization"]
START_PBAR = cfg["start_pbar"]
START_R = cfg["start_r"]
START_WEIGHTS = cfg["start_weights"]
PBAR_MIN = cfg["pbar_min"]
PBAR_MAX = cfg["pbar_max"]
R_MIN = cfg["r_min"]
R_MAX = cfg["r_max"]

# Raw heuristic objective: M_RSR + lambda_track * epsilon_90 + lambda_qos * qos_mean.
# These are NOT paper beta/rho constants. They only guide the raw model's optimizer.
OBJECTIVE_WEIGHTS = "1,30,200"
ITERATIONS = 1500
LR = 0.0025

print("CASE:", CASE)
print("WORKLOAD_CONFIG:", WORKLOAD_CONFIG)
print("START:", START_PBAR, START_R, START_WEIGHTS)
print("P/R bounds:", PBAR_MIN, PBAR_MAX, R_MIN, R_MAX)
print("Raw objective weights:", OBJECTIVE_WEIGHTS)

## 6. Run predict-one only

This only asks the model what it predicts at the starting point.

In [ ]:
import subprocess, json, sys
from pathlib import Path

PREDICT_OUT = RESULTS_DIR / f"predict_one_{CASE}_{TARGET_FAMILY}_{TARGET_MODE}.json"
cmd = [
    sys.executable, "am_unified_predict_one.py",
    "--model-file", str(MODEL_FILE),
    "--norm-source-results-csv", str(RESULTS_CSV),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--target-family", TARGET_FAMILY,
    "--target-mode", TARGET_MODE,
    "--raw-qos-aggregation", RAW_QOS_AGGREGATION,
    "--use-norm-cost", USE_NORM_COST,
    "--use-norm-pr", USE_NORM_PR,
    "--server-count", str(SERVER_COUNT),
    "--utilization", str(UTILIZATION),
    "--pbar-kw-per-server", str(START_PBAR),
    "--r-kw-per-server", str(START_R),
    "--weights", START_WEIGHTS,
    "--objective-weights", OBJECTIVE_WEIGHTS,
    "--device", DEVICE,
    "--out-json", str(PREDICT_OUT),
    "--wandb-mode", WANDB_MODE,
]
if USE_WANDB and WANDB_MODE != "disabled":
    cmd += ["--wandb-project", WANDB_PROJECT, "--wandb-entity", WANDB_ENTITY, "--wandb-run-name", f"predict-one-raw-{CASE}"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Saved:", PREDICT_OUT)

## 7. Run end-to-end model optimization + FlexDC validation

This uses the raw model to optimize P/R/weights inside the explicit local P/R box, then validates the start and selected configurations in FlexDC.

In [ ]:
import subprocess, sys

E2E_DIR = RESULTS_DIR / f"e2e_raw_{CASE}_N{SERVER_COUNT}_U{UTILIZATION:.2f}"
cmd = [
    sys.executable, "am_unified_end_to_end_eval_raw_v2.py",
    "--model-file", str(MODEL_FILE),
    "--norm-source-results-csv", str(RESULTS_CSV),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--target-family", TARGET_FAMILY,
    "--target-mode", TARGET_MODE,
    "--raw-qos-aggregation", RAW_QOS_AGGREGATION,
    "--use-norm-cost", USE_NORM_COST,
    "--use-norm-pr", USE_NORM_PR,
    "--server-count", str(SERVER_COUNT),
    "--utilization", str(UTILIZATION),
    "--start-pbar-kw-per-server", str(START_PBAR),
    "--start-r-kw-per-server", str(START_R),
    "--start-weights", START_WEIGHTS,
    "--iterations", str(ITERATIONS),
    "--lr", str(LR),
    "--objective-weights", OBJECTIVE_WEIGHTS,
    "--device", DEVICE,
    "--pbar-min-kw-per-server", str(PBAR_MIN),
    "--pbar-max-kw-per-server", str(PBAR_MAX),
    "--r-lower-kw-per-server", str(R_MIN),
    "--r-max-kw-per-server", str(R_MAX),
    "--flexdc-root", str(FLEXDC_ROOT),
    "--flexdc-python", sys.executable,
    "--run-flexdc",
    "--out-dir", str(E2E_DIR),
    "--wandb-mode", WANDB_MODE,
]
if USE_WANDB and WANDB_MODE != "disabled":
    cmd += ["--wandb-project", WANDB_PROJECT, "--wandb-entity", WANDB_ENTITY, "--wandb-run-name", f"e2e-raw-{CASE}"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Saved E2E_DIR:", E2E_DIR)

## 8. Concise start-vs-selected raw table

This is the table to look at first. It compares model predictions to actual FlexDC validation for the raw targets.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

summary_path = E2E_DIR / "end_to_end_validation_summary.csv"
candidate_path = E2E_DIR / "optimized_candidate.json"
if not summary_path.exists():
    raise FileNotFoundError(summary_path)
summary = pd.read_csv(summary_path)

raw_cols = {
    "M_RSR": ("Predicted_flexdc_M_RSR", "Actual_flexdc_M_RSR"),
    "p90_tracking": ("Predicted_raw_Ctrack_Epsilon_90th", "Actual_raw_Ctrack_Epsilon_90th"),
    "qos_mean": ("Predicted_raw_qos_probability_mean", "Actual_raw_qos_probability_mean"),
}

def safe(row, col, default=np.nan):
    return row[col] if col in row.index else default

def parse_weights(x):
    try:
        vals = json.loads(x) if isinstance(x, str) else x
        return "[" + ", ".join(f"{float(v):.4f}" for v in vals) + "]"
    except Exception:
        return str(x)

rows = []
for _, row in summary.iterrows():
    out = {
        "Configuration": row["Configuration"].replace(" configuration", ""),
        "Pbar": float(row["Pbar_kw_per_server"]),
        "R": float(row["R_kw_per_server"]),
        "Pbar+R": float(row["Pbar_kw_per_server"] + row["R_kw_per_server"]),
        "Pbar-R": float(row["Pbar_kw_per_server"] - row["R_kw_per_server"]),
        "Weights": parse_weights(row.get("Weights", "")),
        "Pred raw objective": safe(row, "Predicted_Optimization_Objective"),
        "Actual raw objective": safe(row, "Actual_Optimization_Objective"),
    }
    for name, (pc, ac) in raw_cols.items():
        pred = safe(row, pc)
        actual = safe(row, ac)
        out[f"Pred {name}"] = pred
        out[f"Actual {name}"] = actual
        out[f"Error {name}"] = pred - actual if pd.notna(pred) and pd.notna(actual) else np.nan
    out["Actual QoS ratio"] = safe(row, "QoS_Violation_Ratio")
    out["Actual max Pj"] = safe(row, "Max_QoS_Delay_Probability")
    out["Tracking pass"] = bool(safe(row, "Ctrack_Epsilon_90th") <= 0.3)
    out["QoS pass"] = bool(safe(row, "QoS_Violation_Ratio") <= 0.1)
    rows.append(out)

report = pd.DataFrame(rows)
num_cols = report.select_dtypes(include=[np.number]).columns
styled = (
    report.style
    .format({c: "{:.4f}" for c in num_cols})
    .hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "normal"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "td", "props": [("border", "1px solid #d1d5db"), ("padding", "6px")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]}])
)

display(Markdown("### Raw model prediction vs FlexDC validation"))
display(styled)

raw_table_csv = E2E_DIR / "concise_raw_validation_table.csv"
report.to_csv(raw_table_csv, index=False)
print("Saved:", raw_table_csv)

## 9. Constants playground for paper-form objective

This recomputes the **actual** paper-form objective from FlexDC raw validation outputs for any constants. The predicted objective is approximate because the current raw model predicts only mean QoS probability, not the full per-job-type vector.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Change these freely.
BETA = 20.0
RHO = 2.0
DELTA = 0.1
PSI = 1.0
MU = 10.0
GAMMA = 0.3

J = 4  # current W1/W2 workloads have 4 job types

def softplus(x):
    return np.logaddexp(0.0, x)

def parse_probs(x):
    if pd.isna(x) or str(x).strip() == "":
        return None
    return np.asarray(json.loads(x), dtype=float)

rows = []
for _, row in summary.iterrows():
    label = row["Configuration"].replace(" configuration", "")
    actual_m = row.get("Actual_flexdc_M_RSR", np.nan)
    actual_eps = row.get("Actual_raw_Ctrack_Epsilon_90th", np.nan)
    actual_qmean = row.get("Actual_raw_qos_probability_mean", np.nan)
    probs = parse_probs(row.get("QoS_Delay_Probabilities", ""))

    pred_m = row.get("Predicted_flexdc_M_RSR", np.nan)
    pred_eps = row.get("Predicted_raw_Ctrack_Epsilon_90th", np.nan)
    pred_qmean = row.get("Predicted_raw_qos_probability_mean", np.nan)

    actual_ctrack = PSI * softplus(MU * (actual_eps - GAMMA))
    actual_cqos_exact = BETA * np.sum(softplus(RHO * (probs - DELTA))) if probs is not None else np.nan
    actual_paper_obj = actual_m + actual_ctrack + actual_cqos_exact

    pred_ctrack = PSI * softplus(MU * (pred_eps - GAMMA))
    pred_cqos_approx = BETA * J * softplus(RHO * (pred_qmean - DELTA))
    pred_paper_obj_approx = pred_m + pred_ctrack + pred_cqos_approx

    rows.append({
        "Configuration": label,
        "Actual M_RSR": actual_m,
        "Actual p90": actual_eps,
        "Actual QoS mean": actual_qmean,
        "Actual max Pj": np.max(probs) if probs is not None else np.nan,
        "Actual Ctrack": actual_ctrack,
        "Actual CQoS exact": actual_cqos_exact,
        "Actual paper objective": actual_paper_obj,
        "Pred M_RSR": pred_m,
        "Pred p90": pred_eps,
        "Pred QoS mean": pred_qmean,
        "Pred Ctrack": pred_ctrack,
        "Pred CQoS approx": pred_cqos_approx,
        "Pred paper objective approx": pred_paper_obj_approx,
    })

const_report = pd.DataFrame(rows)
num_cols = const_report.select_dtypes(include=[np.number]).columns
styled = (
    const_report.style
    .format({c: "{:.4f}" for c in num_cols})
    .hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "normal"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "td", "props": [("border", "1px solid #d1d5db"), ("padding", "6px")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]}])
)

display(Markdown(f"### Paper-form objective with beta={BETA}, rho={RHO}, delta={DELTA}, psi={PSI}, mu={MU}, gamma={GAMMA}"))
display(styled)

const_csv = E2E_DIR / "paper_objective_constants_playground.csv"
const_report.to_csv(const_csv, index=False)
print("Saved:", const_csv)
print("Note: predicted CQoS/objective are approximate because current raw model predicts QoS mean, not per-job Pj vector.")

## 10. Optional: inspect optimization trajectory

In [ ]:
traj_path = E2E_DIR / "optimization_trajectory.csv"
traj = pd.read_csv(traj_path)
cols = [c for c in traj.columns if c in [
    "Iteration", "Pbar_kw_per_server", "R_kw_per_server", "Predicted_Optimization_Objective",
    "Predicted_flexdc_M_RSR", "Predicted_raw_Ctrack_Epsilon_90th", "Predicted_raw_qos_probability_mean"
]]
display(traj[cols].head())
display(traj[cols].tail())